[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/086.%20The%20Capacitated%20Facility%20Location%20Problem%20%28CFLP%29/P86-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 86. The Capacitated Facility Location Problem (CFLP)

## Tier 3: The Advanced Algorithm (Differential Evolution)

### Key Assumptions
- Customer demand is deterministic (single scenario for metaheuristic simplicity)
- Population-based search using real-valued encoding for facility priorities
- Evolutionary operators: mutation, crossover, and selection
- Adaptive parameter control for F (scaling factor) and CR (crossover rate)
- Greedy customer assignment for fitness evaluation

### Approach (Step-by-Step)

Differential Evolution (DE) is a powerful population-based metaheuristic inspired by natural evolution. Unlike traditional genetic algorithms, DE uses **vector differences** for mutation, making it particularly effective for continuous optimization problems.

**Algorithm Structure:**
1. **Encoding**: Real-valued vector of facility priorities (length = number of facilities)
2. **Population**: N individuals with random priority vectors
3. **Evolution Cycle** (for each generation):
   - **Mutation**: Create mutant vector using three random individuals: **v = x_r1 + F × (x_r2 - x_r3)**
   - **Crossover**: Mix target and mutant vectors with probability CR
   - **Selection**: Keep better solution between target and trial vectors
4. **Decoding**: Convert priority vector to binary facility opening decisions
5. **Fitness**: Evaluate total cost using greedy customer assignment

### What to Look for in the Results
- Convergence behavior and population diversity over generations
- Effect of DE parameters (F, CR, population size) on solution quality
- Comparison with Tier 1 (optimal) and Tier 2 (beam search) solutions
- Trade-off between exploration (diversity) and exploitation (convergence)

### Concrete Example

We'll apply DE to a challenging instance:
- **10 potential facilities** (larger than previous tiers)
- **8 customers** with deterministic demand
- **Population size**: 50 individuals
- **Generations**: 200 evolution cycles
- **Parameters**: F=0.8, CR=0.9 (standard DE settings)

This demonstrates how DE handles complex search spaces and finds high-quality solutions.

In [1]:
from typing import Tuple, List, Dict, Set
from itertools import combinations

# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple
import random
import time
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Facility:
    """Represents a potential facility location"""
    id: int
    name: str
    fixed_cost: float
    capacity: float
    x_coord: float
    y_coord: float

@dataclass
class Customer:
    """Represents a customer location"""
    id: int
    name: str
    demand: float
    x_coord: float
    y_coord: float

@dataclass
class DEIndividual:
    """Represents an individual in the DE population"""
    priorities: np.ndarray  # Real-valued priority vector
    fitness: float          # Total cost (lower is better)
    open_facilities: set    # Decoded facility opening decisions

In [ ]:
# Define larger instance for DE demonstration

# 10 potential facilities with diverse characteristics
facilities = [
    Facility(1, "North Mega Hub", 150000, 1200, 15, 85),   # Very large capacity, high cost
    Facility(2, "Central DC", 95000, 800, 45, 50),         # Medium capacity, medium cost
    Facility(3, "South DC", 75000, 600, 85, 15),         # Small-medium capacity, low cost
    Facility(4, "East Facility", 90000, 750, 95, 65),     # Medium capacity, medium-high cost
    Facility(5, "West Facility", 80000, 700, 5, 35),      # Medium capacity, medium cost
    Facility(6, "Northeast DC", 110000, 850, 75, 90),    # Large capacity, high cost
    Facility(7, "Southwest DC", 70000, 650, 25, 10),     # Small-medium capacity, low cost
    Facility(8, "Central Hub", 130000, 1000, 50, 55),    # Large capacity, high cost
    Facility(9, "Northwest DC", 85000, 720, 20, 75),     # Medium capacity, medium-high cost
    Facility(10, "Southeast DC", 78000, 680, 80, 25)     # Medium capacity, medium cost
]

# 8 customers with varying demand levels
customers = [
    Customer(1, "Store A", 350, 20, 80),
    Customer(2, "Store B", 480, 40, 70),
    Customer(3, "Store C", 320, 70, 30),
    Customer(4, "Store D", 420, 90, 60),
    Customer(5, "Store E", 380, 30, 20),
    Customer(6, "Store F", 450, 60, 90),
    Customer(7, "Store G", 290, 85, 40),
    Customer(8, "Store H", 410, 55, 75)
]

print("Differential Evolution Instance Summary:")
print(f"Facilities: {len(facilities)}")
print(f"Customers: {len(customers)}")
print(f"Total Demand: {sum(c.demand for c in customers)}")
print(f"Total Capacity: {sum(f.capacity for f in facilities)}")
print(f"Search Space Size: 2^{len(facilities)} = {2**len(facilities):,} possible solutions")

Differential Evolution Instance Summary:
Facilities: 10
Customers: 8
Total Demand: 3100
Total Capacity: 7950
Search Space Size: 2^10 = 1,024 possible solutions


In [ ]:
# Calculate transportation costs
def calculate_transport_cost(facility: Facility, customer: Customer) -> float:
    """Calculate transportation cost based on Euclidean distance"""
    distance = np.sqrt((facility.x_coord - customer.x_coord)**2 + 
                      (facility.y_coord - customer.y_coord)**2)
    cost_per_unit = 7.5  # $/unit/distance
    return distance * cost_per_unit

# Build transportation cost matrix
transport_costs = {}
for facility in facilities:
    for customer in customers:
        transport_costs[(facility.id, customer.id)] = calculate_transport_cost(facility, customer)

print("Transportation cost matrix calculated successfully")
print(f"Cost range: ${min(transport_costs.values()):.2f} - ${max(transport_costs.values()):.2f} per unit")

Transportation cost matrix calculated successfully
Cost range: $37.50 - $689.43 per unit


In [ ]:
def decode_solution(priorities: np.ndarray, threshold: float = 0.5) -> set:
    """
    Decode real-valued priority vector to binary facility opening decisions
    
    Args:
        priorities: Real-valued vector of facility priorities
        threshold: Priority threshold for opening facilities
    
    Returns:
        Set of facility IDs to open
    """
    open_facilities = set()
    for i, priority in enumerate(priorities):
        if priority > threshold:
            open_facilities.add(i + 1)  # Facility IDs are 1-indexed
    
    # Ensure at least one facility is open
    if not open_facilities:
        # Open facility with highest priority
        best_facility = np.argmax(priorities) + 1
        open_facilities.add(best_facility)
    
    return open_facilities

def evaluate_fitness(open_facilities: set,
                    facilities: List[Facility],
                    customers: List[Customer],
                    transport_costs: Dict[Tuple[int, int], float]) -> float:
    """
    Evaluate fitness (total cost) of a solution
    Uses greedy customer assignment
    
    Returns:
        Total cost (fixed + transport) or large penalty if infeasible
    """
    
    # Calculate fixed costs
    fixed_cost = sum(f.fixed_cost for f in facilities if f.id in open_facilities)
    
    # Greedy customer assignment
    transport_cost = 0
    facility_loads = {f.id: 0 for f in facilities if f.id in open_facilities}
    
    # Sort customers by demand (largest first)
    sorted_customers = sorted(customers, key=lambda c: c.demand, reverse=True)
    
    for customer in sorted_customers:
        # Find cheapest open facility with available capacity
        best_facility = None
        best_cost = float('inf')
        
        for facility_id in open_facilities:
            facility = next(f for f in facilities if f.id == facility_id)
            
            # Check capacity constraint
            if facility_loads[facility_id] + customer.demand <= facility.capacity:
                cost = transport_costs[(facility_id, customer.id)]
                if cost < best_cost:
                    best_cost = cost
                    best_facility = facility_id
        
        # If no facility can serve this customer, apply penalty
        if best_facility is None:
            return 1e9  # Large penalty for infeasible solutions
        
        # Assign customer to best facility
        transport_cost += best_cost * customer.demand
        facility_loads[best_facility] += customer.demand
    
    return fixed_cost + transport_cost

In [ ]:
try:
    def initialize_population(pop_size: int, num_facilities: int) -> List[DEIndividual]:
        """
        Initialize DE population with random priority vectors
        
        Args:
            pop_size: Number of individuals in population
            num_facilities: Number of potential facilities
        
        Returns:
            List of initialized individuals
        """
        population = []
        
        for _ in range(pop_size):
            # Random priority vector in [0, 1]
            priorities = np.random.random(num_facilities)
            
            # Decode to binary solution
            open_facilities = decode_solution(priorities)
            
            # Evaluate fitness
            fitness = evaluate_fitness(open_facilities, facilities, customers, transport_costs)
            
            individual = DEIndividual(priorities, fitness, open_facilities)
            population.append(individual)
        
        return population
    
    def differential_evolution(pop_size: int = 50,
                              generations: int = 200,
                              F: float = 0.8,
                              CR: float = 0.9,
                              threshold: float = 0.5) -> Dict:
        """
        Differential Evolution algorithm for CFLP
        
        Args:
            pop_size: Population size (N)
            generations: Number of evolution cycles
            F: Scaling factor for mutation (typically 0.5-1.0)
            CR: Crossover rate (typically 0.8-1.0)
            threshold: Priority threshold for decoding
        
        Returns:
            Dictionary containing solution and evolution statistics
        """
        
        start_time = time.time()
        num_facilities = len(facilities)
        
        # Initialize population
        population = initialize_population(pop_size, num_facilities)
        
        # Track evolution statistics
        evolution_history = {
            'best_fitness': [],
            'avg_fitness': [],
            'worst_fitness': [],
            'diversity': []
            'generation': []
        }
        
        best_individual = min(population, key=lambda ind: ind.fitness)
        
        print(f"Starting DE with {pop_size} individuals for {generations} generations")
        print(f"Parameters: F={F}, CR={CR}, threshold={threshold}")
        print(f"Initial best fitness: ${best_individual.fitness:,.2f}")
        
        # Evolution loop
        for gen in range(generations):
            new_population = []
            
            for i, target in enumerate(population):
                # Mutation: Select 3 random distinct individuals
                candidates = [j for j in range(pop_size) if j != i]
                r1, r2, r3 = random.sample(candidates, 3)
                
                x_r1 = population[r1].priorities
                x_r2 = population[r2].priorities
                x_r3 = population[r3].priorities
                
                # Create mutant vector
                mutant = x_r1 + F * (x_r2 - x_r3)
                
                # Ensure mutant values are in [0, 1]
                mutant = np.clip(mutant, 0, 1)
                
                # Crossover: Create trial vector
                trial = target.priorities.copy()
                for j in range(num_facilities):
                    if random.random() < CR or j == num_facilities - 1:
                        trial[j] = mutant[j]
                
                # Decode and evaluate trial solution
                trial_open_facilities = decode_solution(trial, threshold)
                trial_fitness = evaluate_fitness(trial_open_facilities, facilities, customers, transport_costs)
                
                # Selection: Keep better solution
                if trial_fitness < target.fitness:
                    new_individual = DEIndividual(trial, trial_fitness, trial_open_facilities)
                    new_population.append(new_individual)
                else:
                    new_population.append(target)
            
            population = new_population
            
            # Update statistics
            fitness_values = [ind.fitness for ind in population]
            current_best = min(fitness_values)
            current_avg = np.mean(fitness_values)
            current_worst = max(fitness_values)
            
            # Calculate diversity (average pairwise distance)
            diversity = 0
            for i in range(pop_size):
                for j in range(i + 1, pop_size):
                    diversity += np.linalg.norm(population[i].priorities - population[j].priorities)
            diversity /= (pop_size * (pop_size - 1) / 2)
            
            evolution_history['best_fitness'].append(current_best)
            evolution_history['avg_fitness'].append(current_avg)
            evolution_history['worst_fitness'].append(current_worst)
            evolution_history['diversity'].append(diversity)
            evolution_history['generation'].append(gen + 1)
            
            # Update best individual
            generation_best = min(population, key=lambda ind: ind.fitness)
            if generation_best.fitness < best_individual.fitness:
                best_individual = generation_best
            
            # Progress reporting
            if (gen + 1) % 50 == 0 or gen == 0:
                print(f"Generation {gen + 1:3d}: Best=${current_best:,.2f}, Avg=${current_avg:,.2f}, Diversity={diversity:.3f}")
        
        end_time = time.time()
        
        return {
            'best_individual': best_individual,
            'final_population': population,
            'evolution_history': evolution_history,
            'computation_time': end_time - start_time,
            'parameters': {
                'pop_size': pop_size,
                'generations': generations,
                'F': F,
                'CR': CR,
                'threshold': threshold
            }
        }
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: invalid syntax. Perhaps you forgot a comma? (<string>, line 59)...]

In [ ]:
try:
    # Run Differential Evolution
    print("="*80)
    print("DIFFERENTIAL EVOLUTION FOR CFLP")
    print("="*80)
    
    # Standard DE parameters
    de_result = differential_evolution(
        pop_size=50,
        generations=200,
        F=0.8,
        CR=0.9,
        threshold=0.5
    )
    
    print(f"\n{'='*80}")
    print("DIFFERENTIAL EVOLUTION RESULTS")
    print(f"{'='*80}")
    print(f"Computation Time: {de_result['computation_time']:.2f} seconds")
    print(f"Best Solution Cost: ${de_result['best_individual'].fitness:,.2f}")
    print(f"Open Facilities: {sorted(de_result['best_individual'].open_facilities)}")
    
    # Show facility names
    open_facility_names = [
        f.name for f in facilities if f.id in de_result['best_individual'].open_facilities
    ]
    print(f"Facility Names: {open_facility_names}")
    print(f"Number of Open Facilities: {len(de_result['best_individual'].open_facilities)}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'differential_evolution' is not defined...]

In [ ]:
try:
    # Parameter sensitivity analysis
    print("\n" + "="*80)
    print("PARAMETER SENSITIVITY ANALYSIS")
    print("="*80)
    
    # Test different parameter combinations
    parameter_tests = [
        {'pop_size': 30, 'generations': 150, 'F': 0.5, 'CR': 0.8, 'threshold': 0.5},
        {'pop_size': 50, 'generations': 200, 'F': 0.8, 'CR': 0.9, 'threshold': 0.5},  # Base case
        {'pop_size': 70, 'generations': 250, 'F': 0.9, 'CR': 0.95, 'threshold': 0.5},
        {'pop_size': 50, 'generations': 200, 'F': 0.8, 'CR': 0.9, 'threshold': 0.4},
        {'pop_size': 50, 'generations': 200, 'F': 0.8, 'CR': 0.9, 'threshold': 0.6},
    ]
    
    sensitivity_results = []
    
    for i, params in enumerate(parameter_tests):
        print(f"\nTest {i+1}: F={params['F']}, CR={params['CR']}, threshold={params['threshold']}, pop_size={params['pop_size']}")
        
        result = differential_evolution(**params)
        
        sensitivity_results.append({
            'test': i+1,
            'F': params['F'],
            'CR': params['CR'],
            'threshold': params['threshold'],
            'pop_size': params['pop_size'],
            'generations': params['generations'],
            'best_cost': result['best_individual'].fitness,
            'computation_time': result['computation_time'],
            'open_facilities': len(result['best_individual'].open_facilities)
        })
        
        print(f"  Best Cost: ${result['best_individual'].fitness:,.2f}")
        print(f"  Time: {result['computation_time']:.2f}s")
        print(f"  Open Facilities: {len(result['best_individual'].open_facilities)}")
    
    # Create sensitivity analysis table
    sensitivity_df = pd.DataFrame(sensitivity_results)
    print(f"\n{'='*80}")
    print("SENSITIVITY ANALYSIS SUMMARY")
    print(f"{'='*80}")
    print(sensitivity_df[['test', 'F', 'CR', 'threshold', 'best_cost', 'computation_time', 'open_facilities']].round(2))
    
    # Find best parameter combination
    best_test = min(sensitivity_results, key=lambda x: x['best_cost'])
    print(f"\nBest Parameter Combination: Test {best_test['test']}")
    print(f"Best Cost: ${best_test['best_cost']:,.2f}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'differential_evolution' is not defined...]

In [ ]:
try:
    # Detailed analysis of best solution
    best_individual = de_result['best_individual']
    open_facilities = [f for f in facilities if f.id in best_individual.open_facilities]
    
    print("\n" + "="*80)
    print("DETAILED ANALYSIS OF BEST DE SOLUTION")
    print("="*80)
    
    # Priority vector analysis
    print("\nPriority Vector Analysis:")
    for i, (facility, priority) in enumerate(zip(facilities, best_individual.priorities)):
        status = "OPEN" if facility.id in best_individual.open_facilities else "CLOSED"
        print(f"  {facility.name}: Priority={priority:.3f}, Status={status}")
    
    # Cost breakdown
    fixed_costs = sum(f.fixed_cost for f in open_facilities)
    transport_costs = best_individual.fitness - fixed_costs
    
    print(f"\nCost Breakdown:")
    print(f"  Fixed Costs: ${fixed_costs:,}")
    print(f"  Transport Costs: ${transport_costs:,}")
    print(f"  Total Cost: ${best_individual.fitness:,}")
    
    # Customer assignments (greedy)
    facility_loads = {f.id: 0 for f in open_facilities}
    assignments = {}
    
    sorted_customers = sorted(customers, key=lambda c: c.demand, reverse=True)
    
    print(f"\nCustomer Assignments:")
    for customer in sorted_customers:
        # Find assignment (same logic as evaluate_fitness)
        best_facility_id = None
        best_cost = float('inf')
        
        for facility in open_facilities:
            if facility_loads[facility.id] + customer.demand <= facility.capacity:
                cost = transport_costs[(facility.id, customer.id)]
                if cost < best_cost:
                    best_cost = cost
                    best_facility_id = facility.id
        
        facility_name = next(f.name for f in open_facilities if f.id == best_facility_id)
        assignments[customer.id] = facility_name
        facility_loads[best_facility_id] += customer.demand
        
        print(f"  {customer.name} (demand {customer.demand}) → {facility_name} (cost ${best_cost * customer.demand:.2f})")
    
    print(f"\nFacility Utilization:")
    for facility in open_facilities:
        utilization = (facility_loads[facility.id] / facility.capacity) * 100
        print(f"  {facility.name}: {facility_loads[facility.id]}/{facility.capacity} ({utilization:.1f}%)")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'de_result' is not defined...]

In [ ]:
try:
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Differential Evolution CFLP Analysis', fontsize=16, fontweight='bold')
    
    # 1. Convergence plot
    ax1 = axes[0, 0]
    history = de_result['evolution_history']
    ax1.plot(history['generation'], history['best_fitness'], 'b-', linewidth=2, label='Best')
    ax1.plot(history['generation'], history['avg_fitness'], 'g--', linewidth=1, label='Average')
    ax1.plot(history['generation'], history['worst_fitness'], 'r:', linewidth=1, label='Worst')
    ax1.set_xlabel('Generation')
    ax1.set_ylabel('Total Cost ($)')
    ax1.set_title('Convergence Behavior')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Population diversity
    ax2 = axes[0, 1]
    ax2.plot(history['generation'], history['diversity'], 'purple', linewidth=2)
    ax2.set_xlabel('Generation')
    ax2.set_ylabel('Population Diversity')
    ax2.set_title('Population Diversity Over Time')
    ax2.grid(True, alpha=0.3)
    
    # 3. Parameter sensitivity
    ax3 = axes[0, 2]
    test_names = [f"Test {r['test']}" for r in sensitivity_results]
    costs = [r['best_cost'] for r in sensitivity_results]
    colors = ['red' if c == min(costs) else 'steelblue' for c in costs]
    ax3.bar(test_names, costs, color=colors, alpha=0.7)
    ax3.set_xlabel('Parameter Test')
    ax3.set_ylabel('Best Cost ($)')
    ax3.set_title('Parameter Sensitivity')
    ax3.grid(True, alpha=0.3)
    plt.setp(ax3.get_xticklabels(), rotation=45)
    
    # 4. Facility locations and assignments
    ax4 = axes[1, 0]
    # Plot all facilities
    for facility in facilities:
        color = 'red' if facility.id in best_individual.open_facilities else 'lightgray'
        marker = 's' if facility.id in best_individual.open_facilities else 's'
        size = 400 if facility.id in best_individual.open_facilities else 100
        ax4.scatter(facility.x_coord, facility.y_coord, c=color, marker=marker, s=size,
                   edgecolors='black', linewidths=2)
        ax4.annotate(f"{facility.name}", (facility.x_coord, facility.y_coord),
                    xytext=(5, 5), textcoords='offset points', fontsize=7)
    
    # Plot customers
    for customer in customers:
        ax4.scatter(customer.x_coord, customer.y_coord, c='blue', marker='o', s=120,
                   edgecolors='black', linewidths=1)
        ax4.annotate(f"{customer.name}\n({customer.demand})", (customer.x_coord, customer.y_coord),
                    xytext=(5, 5), textcoords='offset points', fontsize=7)
    
    ax4.set_xlabel('X Coordinate')
    ax4.set_ylabel('Y Coordinate')
    ax4.set_title('Best DE Solution: Facility Locations')
    ax4.grid(True, alpha=0.3)
    
    # 5. Priority vector distribution
    ax5 = axes[1, 1]
    facility_names = [f.name[:8] for f in facilities]  # Truncate for display
    priorities = best_individual.priorities
    colors = ['red' if i+1 in best_individual.open_facilities else 'lightblue' for i in range(len(facilities))]
    ax5.barh(facility_names, priorities, color=colors, alpha=0.7)
    ax5.axvline(x=0.5, color='black', linestyle='--', alpha=0.5, label='Threshold')
    ax5.set_xlabel('Priority Value')
    ax5.set_title('Facility Priority Vector')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # 6. Cost breakdown
    ax6 = axes[1, 2]
    cost_breakdown = [fixed_costs, transport_costs]
    labels = ['Fixed Costs', 'Transport Costs']
    colors_pie = ['#ff7f0e', '#1f77b4']
    ax6.pie(cost_breakdown, labels=labels, colors=colors_pie, autopct='%1.1f%%', startangle=90)
    ax6.set_title('Cost Structure Breakdown')
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'de_result' is not defined...]

In [ ]:
try:
    # Comparison with other tiers (conceptual)
    print("\n" + "="*80)
    print("ALGORITHM COMPARISON")
    print("="*80)
    
    # Conceptual comparison based on typical performance
    comparison_data = [
        {
            'Tier': 'Tier 1 (MIP)',
            'Solution Quality': 'Optimal',
            'Computation Time': 'Minutes to Hours',
            'Scalability': 'Poor (≤20 facilities)',
            'Memory Usage': 'High',
            'Deterministic': 'Yes'
        },
        {
            'Tier': 'Tier 2 (Beam Search)',
            'Solution Quality': 'Good (5-15% from optimal)',
            'Computation Time': 'Seconds',
            'Scalability': 'Good (≤100 facilities)',
            'Memory Usage': 'Low',
            'Deterministic': 'Yes'
        },
        {
            'Tier': 'Tier 3 (DE)',
            'Solution Quality': 'Very Good (2-8% from optimal)',
            'Computation Time': 'Seconds to Minutes',
            'Scalability': 'Excellent (≤200 facilities)',
            'Memory Usage': 'Medium',
            'Deterministic': 'No (stochastic)'
        }
    ]
    
    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))
    
    print(f"\nDE Solution Quality Assessment:")
    print(f"Best DE Cost: ${best_individual.fitness:,.2f}")
    print(f"Open Facilities: {len(best_individual.open_facilities)}")
    print(f"Computation Time: {de_result['computation_time']:.2f} seconds")
    print(f"Convergence: Achieved in {len(history['best_fitness'])} generations")
    print(f"Final Diversity: {history['diversity'][-1]:.3f}")
    
    print(f"\nKey DE Advantages Demonstrated:")
    print(f"✅ Handles large search spaces (2^10 = 1,024 possible solutions)")
    print(f"✅ Continuous optimization with real-valued encoding")
    print(f"✅ Effective balance of exploration and exploitation")
    print(f"✅ Parameter tuning capability for problem-specific optimization")
    print(f"✅ Population diversity tracking for convergence analysis")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'best_individual' is not defined...]

### Why This Tier Exists vs Previous Tiers

**Tier 3 (Differential Evolution) addresses limitations of both Tier 1 and Tier 2:**

**vs Tier 1 (MIP):**
- **Scalability**: DE handles 50-200+ facilities where MIP becomes intractable
- **Memory efficiency**: Population-based search vs branch-and-bound memory explosion
- **Anytime capability**: Can stop early and return best solution found
- **Parallelizable**: Population evaluation can be distributed

**vs Tier 2 (Beam Search):**
- **Global search**: DE explores entire search space vs beam's limited paths
- **Adaptive operators**: Mutation and crossover adapt to problem landscape
- **Population diversity**: Maintains multiple solutions vs single beam path
- **Continuous optimization**: Real-valued encoding vs discrete decisions

**Key Innovations:**
- **Differential mutation**: Uses vector differences for effective exploration
- **Self-adaptation**: Parameters can be adapted during evolution
- **Landscape navigation**: Better at escaping local optima than greedy methods
- **Statistical analysis**: Rich population statistics for convergence monitoring

### Pros and Cons

**Advantages:**
✅ **Excellent scalability** - Handles very large problem instances
✅ **Global optimization** - Effective at finding high-quality solutions
✅ **Few parameters** - Only F, CR, and population size to tune
✅ **Robust performance** - Works well across diverse problem types
✅ **Rich feedback** - Population diversity and convergence statistics
✅ **Parallelizable** - Population evaluation can be distributed

**Disadvantages:**
❌ **No optimality guarantee** - Stochastic nature of search
❌ **Parameter sensitivity** - Performance depends on F, CR settings
❌ **Computation time** - Slower than greedy methods for small problems
❌ **Decoding complexity** - Requires priority-to-binary conversion
❌ **Stochastic results** - Different solutions on different runs

### When to Use This Tier

**Ideal for:**
- **Large-scale problems** (50-200+ facilities) where exact methods fail
- **Complex search spaces** with many local optima
- **When near-optimal solutions** are acceptable for faster computation
- **Research and development** testing multiple solution approaches
- **Problems with smooth fitness landscapes** amenable to continuous optimization
- **Multi-objective extensions** where DE can be adapted

**Not ideal for:**
- **Small problems** (<10 facilities) where exact methods are fast
- **Real-time decisions** requiring instantaneous solutions
- **Problems requiring provable optimality** for regulatory compliance
- **Highly discrete problems** without natural continuous encoding
- **Deterministic requirements** where reproducibility is critical